In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Lesson 3: Profiling and Debugging JAX on GPU

## Overview

When JAX-on-GPU code is slow, the Python source usually does not tell you why. The bottleneck might be compilation, asynchronous timing mistakes, host-device transfers, small batches, memory pressure, or a CUDA-level issue.

**What you'll do:**

1. Verify that arrays and computation are on the GPU.
2. Separate first-call compilation time from cached execution time.
3. Time JAX correctly with `block_until_ready()`.
4. Capture one JAX profiler trace and open it in XProf or TensorBoard.
5. Diagnose common problems: host transfers, too many small operations, inefficient batches, and memory pressure.
6. Capture one required Nsight Systems report for the CUDA-level timeline.

The goal is not to memorize every profiler feature. The goal is to learn where to start when GPU utilization is poor or a model runs out of memory.

## Profiling workflow

A useful way to approach JAX performance debugging is to move from simple checks to deeper tools. First, make sure your timing is real by blocking until GPU work finishes. Then capture a JAX profiler trace so you can see compilation, host activity, and device execution together. Next, open the trace in XProf or TensorBoard to inspect timelines, memory, graphs, and operation statistics. When you need the CUDA-level view, use Nsight Systems to see streams, kernels, memory copies, library calls, and communication.

<div style="font-family: Arial, sans-serif; max-width: 980px; line-height: 1.35; display: grid; grid-template-columns: 1fr 28px 1fr 28px 1fr 28px 1fr; gap: 10px; align-items: stretch;">

  <div style="grid-column: 1; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">1. Measure</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Block before timing</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">Use <code>block_until_ready()</code> so you measure GPU execution, not queueing.</div>
  </div>

  <div style="grid-column: 2; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #57606a;">&#8594;</div>

  <div style="grid-column: 3; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">2. Trace</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">JAX profiler</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">Capture JAX, host, and device activity with <code>jax.profiler</code>.</div>
  </div>

  <div style="grid-column: 4; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #57606a;">&#8594;</div>

  <div style="grid-column: 5; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">3. View</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">XProf / TensorBoard</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">Use trace viewer, memory viewer, graph viewer, and op stats.</div>
  </div>

  <div style="grid-column: 6; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #57606a;">&#8594;</div>

  <div style="grid-column: 7; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #1a7f37; background: #dcffe4; display: inline-block; padding: 2px 8px; border-radius: 12px;">4. Deep dive</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Nsight Systems</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">Required CUDA timeline for streams, API calls, kernels, memcopies, and NCCL.</div>
  </div>

</div>

### What to look for

When you open a trace, do not try to understand every event at once. Start by scanning for a few common visual patterns. Compile spans tell you whether time is going into XLA compilation instead of execution. Empty gaps on GPU rows often mean the host is not feeding the device quickly enough. Transfer activity can point to accidental host-device synchronization, such as logging with `float(loss)` or converting arrays to NumPy. Memory peaks help you spot batches, activations, or temporary buffers that may be pushing the GPU close to its limit.

<div style="font-family: Arial, sans-serif; max-width: 980px; line-height: 1.35; display: grid; grid-template-columns: 1fr 28px 1fr 28px 1fr 28px 1fr; gap: 10px; align-items: stretch;">
  <div style="grid-column: 1; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #9a6700; background: #fff8c5; display: inline-block; padding: 2px 8px; border-radius: 12px;">Look for</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Compilation</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">XLA compile spans before or between steps.</div>
  </div>

  <div style="grid-column: 3; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #9a6700; background: #fff8c5; display: inline-block; padding: 2px 8px; border-radius: 12px;">Look for</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Host gaps</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">White space on GPU rows while Python or data loading is busy.</div>
  </div>

  <div style="grid-column: 5; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #9a6700; background: #fff8c5; display: inline-block; padding: 2px 8px; border-radius: 12px;">Look for</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Transfers</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;"><code>float(loss)</code>, <code>.item()</code>, printing, NumPy conversion.</div>
  </div>

  <div style="grid-column: 7; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #9a6700; background: #fff8c5; display: inline-block; padding: 2px 8px; border-radius: 12px;">Look for</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Memory pressure</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">Large peaks, allocator limits, transient intermediates.</div>
  </div>
</div>

## Requirements

This notebook assumes the same CUDA-enabled JAX environment as Lessons 1 and 2.

For this lesson you also need:

| Tool | Purpose | Typical install |
| ---- | ------- | --------------- |
| `xprof` | View JAX profiler traces | `pip install xprof` |
| TensorBoard | Optional front-end for the same XProf plugin | `pip install tensorboard xprof` |
| `nsys` | Capture the Nsight Systems CUDA timeline | NVIDIA container or CUDA toolkit |
| `nvtx` | Add named ranges to the Nsight report | `pip install nvtx` |

Standalone XProf is the main path in this notebook; TensorBoard is included as an alternative viewer for environments that already expose port 6006.

## Setup

Import the tools, check the GPU, and define two small display helpers. If this cell fails, fix the environment before continuing; profiling a CPU fallback is misleading.

In [ ]:
import csv
import importlib.util
import io
import os
import pathlib
import shutil
import subprocess
import sys
import tempfile
import textwrap
import time

import jax
import jax.numpy as jnp
import numpy as np
from IPython.display import HTML, Javascript, display


def require_executable(name):
    """Look up `name` on PATH and assert it's found; returns the absolute path or fails fast."""
    path = shutil.which(name)
    assert path, f"Required executable '{name}' not found on PATH."
    return path


XPROF_BIN = require_executable("xprof")
NSYS_BIN = require_executable("nsys")
TENSORBOARD_BIN = shutil.which("tensorboard")
assert importlib.util.find_spec("nvtx"), "Required Python package 'nvtx' is missing. Install with: pip install nvtx"


def show_bars(rows, title, unit="", lower_is_better=False):
    """Render (label, value) pairs as a horizontal bar chart in HTML, scaled to the largest value."""
    max_value = max(float(value) for _, value in rows) or 1.0
    html = ["<div style='font-family: Arial, sans-serif; max-width: 760px;'>"]
    html.append(f"<h4 style='margin: 0 0 8px 0;'>{title}</h4>")
    for label, value in rows:
        width = max(3, 100 * float(value) / max_value)
        html.append(
            "<div style='display:grid; grid-template-columns: 190px 1fr 115px; gap: 8px; "
            "align-items:center; margin: 6px 0;'>"
            f"<div style='font-size:13px;'>{label}</div>"
            "<div style='background:#f6f8fa; border-radius:6px; overflow:hidden; height:22px;'>"
            f"<div style='height:22px; width:{width:.1f}%; background:#0969da;'></div></div>"
            f"<div style='font-size:13px; font-variant-numeric: tabular-nums;'>{value:.3f} {unit}</div>"
            "</div>"
        )
    html.append(f"<div style='font-size:12px; color:#57606a;'>{'Lower' if lower_is_better else 'Higher'} is better.</div></div>")
    display(HTML("".join(html)))


def show_table(headers, rows, title=None, aligns=None):
    """Render rows as an HTML table; `aligns` is an optional per-column list of "left"/"right"/"center"."""
    aligns = aligns or ["left"] * len(headers)
    html = ["<div style='font-family: system-ui; max-width: 980px;'>"]
    if title:
        html.append(f"<h4 style='margin-bottom: 8px;'>{title}</h4>")
    html.append("<table style='border-collapse: collapse; width: 100%; font-size: 13px;'>")
    html.append("<thead><tr>")
    for h, a in zip(headers, aligns):
        html.append(f"<th style='text-align:{a}; border-bottom:1px solid #d0d7de; padding:6px;'>{h}</th>")
    html.append("</tr></thead><tbody>")
    for row in rows:
        html.append("<tr>")
        for cell, a in zip(row, aligns):
            html.append(f"<td style='text-align:{a}; border-bottom:1px solid #edf0f2; padding:6px; vertical-align:top; white-space:nowrap;'>{cell}</td>")
        html.append("</tr>")
    html.append("</tbody></table></div>")
    display(HTML("".join(html)))

def show_file_list(root, limit=12):
    """Print files under `root` with their sizes (KB); truncates after `limit` entries."""
    root = pathlib.Path(root)
    files = [p for p in sorted(root.rglob("*")) if p.is_file()]
    for p in files[:limit]:
        print(f"{p.stat().st_size / 1024:8.1f} KB  {p.relative_to(root)}")
    if len(files) > limit:
        print(f"... {len(files) - limit} more files")


devices = jax.devices()
gpu_devices = [d for d in devices if d.platform == "gpu"]

print(f"JAX version:     {jax.__version__}")
print(f"Default backend: {jax.default_backend()}")
print(f"Devices:         {devices}")
print(f"xprof:           {XPROF_BIN}")
print(f"tensorboard: {TENSORBOARD_BIN or 'not found, XProf standalone is enough'}")
print(f"nsys:            {NSYS_BIN}")

assert gpu_devices, f"This lesson assumes a GPU backend. Available devices: {devices}"
print(f"GPU devices:     {gpu_devices}")

## A small training-step workload

We need something realistic enough to profile, but small enough to run quickly. This cell defines a tiny two-layer MLP, a mean-squared-error loss, gradients with `jax.value_and_grad`, and a simple SGD update.

The important bit is the final warmup call. It compiles `train_step` once before later timing cells, so those cells can focus on execution behavior instead of accidentally measuring setup.

In [ ]:
# Keep the dimensions modest so the notebook runs quickly, but large enough to create real GPU work.
BATCH = 256
IN_DIM = 1024
HIDDEN = 1024
OUT_DIM = 256
LR = 1e-3


def init_params(key):
    """Initialize the two-layer MLP weights this lesson profiles."""
    k1, k2 = jax.random.split(key)
    return {
        "w1": jax.random.normal(k1, (IN_DIM, HIDDEN), dtype=jnp.float32) * 0.02,
        "w2": jax.random.normal(k2, (HIDDEN, OUT_DIM), dtype=jnp.float32) * 0.02,
    }


def make_batch(key, batch_size=BATCH):
    """Generate a random (x, target) batch with the standard input/output dims."""
    kx, ky = jax.random.split(key)
    x = jax.random.normal(kx, (batch_size, IN_DIM), dtype=jnp.float32)
    y = jax.random.normal(ky, (batch_size, OUT_DIM), dtype=jnp.float32)
    return x, y


def loss_fn(params, batch):
    """Forward pass plus MSE loss; the function we differentiate and jit below."""
    x, target = batch
    hidden = jax.nn.gelu(x @ params["w1"])
    pred = hidden @ params["w2"]
    return jnp.mean((pred - target) ** 2)


# This is the function we will profile. JIT makes it one compiled training step.
@jax.jit
def train_step(params, batch):
    """One compiled SGD step: `value_and_grad`, then `param := param - LR * grad` over the param tree."""
    loss, grads = jax.value_and_grad(loss_fn)(params, batch)
    params = jax.tree.map(lambda p, g: p - LR * g, params, grads)
    return params, loss


key = jax.random.key(0)
params = init_params(key)
batch = make_batch(jax.random.fold_in(key, 1))

# Warm up once so later timing focuses on execution, not first-call compilation.
params, loss = train_step(params, batch)
jax.block_until_ready((params, loss))  # Wait here so compilation is complete before the next section.

print(f"x shape/device:      {batch[0].shape} on {batch[0].device}")
print(f"target shape/device: {batch[1].shape} on {batch[1].device}")
print(f"w1 shape/device:     {params['w1'].shape} on {params['w1'].device}")
print(f"warmup loss:         {float(loss):.4f}")

## First call vs. cached execution

A jitted function has two very different modes:

* The **first call for a new input signature** traces and compiles, then executes.
* Later calls with the **same shapes and dtypes** reuse the compiled executable.

Changing the batch shape creates a new signature, so JAX has to compile again. This is the root cause of many "my training loop keeps pausing" reports.

In [ ]:
# Clear JAX's in-process compilation cache for this demo only.
# This makes the first-call cost visible even if you rerun the notebook.
jax.clear_caches()

compile_params = init_params(jax.random.key(101))
compile_batch = make_batch(jax.random.key(102), batch_size=BATCH)

# First call: trace + compile + execute.
t0 = time.perf_counter()
compile_params, compile_loss = train_step(compile_params, compile_batch)
jax.block_until_ready((compile_params, compile_loss))
first_ms = (time.perf_counter() - t0) * 1000

# Same shapes and dtypes: execute using the cached compiled executable.
t0 = time.perf_counter()
compile_params, compile_loss = train_step(compile_params, compile_batch)
jax.block_until_ready((compile_params, compile_loss))
cached_ms = (time.perf_counter() - t0) * 1000

# Different batch shape: this triggers another compile.
smaller_batch = make_batch(jax.random.key(103), batch_size=BATCH // 2)
t0 = time.perf_counter()
shape_params, shape_loss = train_step(compile_params, smaller_batch)
jax.block_until_ready((shape_params, shape_loss))
shape_change_ms = (time.perf_counter() - t0) * 1000

print(f"First call, same shape (compile + execute): {first_ms:8.2f} ms")
print(f"Cached call, same shape (execute only):     {cached_ms:8.2f} ms")
print(f"New batch shape (compile + execute):        {shape_change_ms:8.2f} ms")
show_bars(
    [
        ("first call", first_ms),
        ("cached call", cached_ms),
        ("new shape", shape_change_ms),
    ],
    title="Compilation Cost vs. Cached Execution",
    unit="ms",
    lower_is_better=True,
)

# Re-warm the original shape for the rest of the notebook.
params, loss = train_step(params, batch)
jax.block_until_ready((params, loss))

## Time JAX correctly

JAX dispatches GPU work asynchronously. That means Python can return before the GPU has finished. Without `block_until_ready()`, you usually measure how long Python took to enqueue the work, not how long the GPU took to execute it.

This cell times the same training step twice: once incorrectly, and once with a block on the result.

In [ ]:
# Wrong: no block, so this mostly measures dispatch.
t0 = time.perf_counter()
params_dispatch, loss_dispatch = train_step(params, batch)
dispatch_ms = (time.perf_counter() - t0) * 1000

# Right: block on the result pytree.
t0 = time.perf_counter()
params_ready, loss_ready = train_step(params, batch)
jax.block_until_ready((params_ready, loss_ready))
ready_ms = (time.perf_counter() - t0) * 1000

print(f"Dispatch-only timing: {dispatch_ms:.3f} ms")
print(f"Blocked timing:       {ready_ms:.3f} ms")
show_bars(
    [("dispatch only", dispatch_ms), ("block_until_ready", ready_ms)],
    title="Timing the Same JAX Step",
    unit="ms",
    lower_is_better=True,
)

## Pitfall: too many small operations

Another common GPU performance issue is launching many tiny operations from Python. Each small JAX operation has Python dispatch overhead and may lead to small GPU kernels. `jax.jit` helps by letting XLA see the whole chain and fuse or schedule it as a compiled unit.

The two functions below do the same math. The first dispatches the operations from Python. The second compiles the chain.

In [ ]:
SMALL_OP_STEPS = 50
small_x = jnp.ones((4096,), dtype=jnp.float32)


def many_small_ops(x):
    """Un-jitted chain of small operations; each iteration dispatches separately from Python."""
    # Not jitted: each loop iteration dispatches JAX operations from Python.
    y = x
    for _ in range(SMALL_OP_STEPS):
        y = jnp.sin(y) + 0.01 * y
    return y


@jax.jit
def compiled_chain(x):
    """Same operations under `jax.jit` so XLA can fuse them into one compiled program."""
    # Jitted: JAX traces the whole chain, and XLA can optimize it as one compiled computation.
    y = x
    for _ in range(SMALL_OP_STEPS):
        y = jnp.sin(y) + 0.01 * y
    return y


# Warm up the compiled version so we compare execution time, not compilation time.
_ = compiled_chain(small_x).block_until_ready()

# Time the non-jitted chain.
t0 = time.perf_counter()
_ = many_small_ops(small_x).block_until_ready()
small_ops_ms = (time.perf_counter() - t0) * 1000

# Time the compiled chain.
t0 = time.perf_counter()
_ = compiled_chain(small_x).block_until_ready()
compiled_chain_ms = (time.perf_counter() - t0) * 1000

print(f"Many small Python-dispatched ops: {small_ops_ms:8.3f} ms")
print(f"Compiled chain with jax.jit:      {compiled_chain_ms:8.3f} ms")
show_bars(
    [("many small ops", small_ops_ms), ("compiled chain", compiled_chain_ms)],
    title="Too Many Small Operations",
    unit="ms",
    lower_is_better=True,
)

## Capture a JAX profiler trace

A trace tells you what happened over time: when Python was active, when XLA compiled, when the GPU ran kernels, and where there were gaps.

We will mark each training step so it is easy to find in XProf. We also mark `make_batch` and `sgd_update` so the trace has human-readable regions instead of only low-level operation names.

| Annotation | Use it for |
| ---------- | ---------- |
| `StepTraceAnnotation` | Naming repeated steps, such as training iterations. |
| `TraceAnnotation` | Naming a region inside a step, such as batch prep or optimizer update. |
| `annotate_function` | Naming a Python function in the trace. |

In [ ]:
# The trace directory is what XProf and TensorBoard will read.
trace_dir = pathlib.Path(tempfile.mkdtemp(prefix="jax-trace-"))
trace_params = params
trace_key = key

# create_perfetto_link=False avoids a blocking browser prompt in notebook runs.
with jax.profiler.trace(str(trace_dir), create_perfetto_link=False):
    for step_num in range(6):
        with jax.profiler.StepTraceAnnotation("train_step", step_num=step_num):
            with jax.profiler.TraceAnnotation("make_batch"):
                trace_key = jax.random.fold_in(trace_key, step_num + 10)
                trace_batch = make_batch(trace_key)
            with jax.profiler.TraceAnnotation("sgd_update"):
                trace_params, trace_loss = train_step(trace_params, trace_batch)
            jax.block_until_ready((trace_params, trace_loss))  # Make sure device work is inside the trace window.

print(f"Trace directory: {trace_dir}")
show_file_list(trace_dir)

### Open the trace in XProf

In XProf:

1. Open the URL printed below.
2. Choose the run from the **Runs** dropdown.
3. Open **Tools -> trace_viewer**.
4. Find the `train_step` spans.
5. Look for compile spans, GPU gaps, copies, and kernel activity.

In [ ]:
xprof_port = 6007
xprof_proc = subprocess.Popen(
    [XPROF_BIN, "--port", str(xprof_port), str(trace_dir)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)
xprof_url = f"http://localhost:{xprof_port}"
print(f"XProf is running at {xprof_url} (PID {xprof_proc.pid})")
print(f"Stop it later with: kill {xprof_proc.pid}")
display(Javascript(f'window.open("{xprof_url}", "_blank");'))
display(HTML(f'<a href="{xprof_url}" target="_blank" rel="noopener">Open XProf</a>'))

### Alternative viewer: TensorBoard profile tab

The same trace directory can be opened through TensorBoard when the XProf plugin is installed. In current JAX docs, XProf is the underlying profiler UI and TensorBoard becomes another front-end for it.

Use this when your environment already standardizes on TensorBoard, or when a managed notebook exposes TensorBoard more easily than arbitrary ports.

In [ ]:
if TENSORBOARD_BIN is None:
    print("TensorBoard executable not found. XProf standalone above is enough for this lesson.")
    print("To add TensorBoard later: pip install tensorboard xprof")
else:
    tb_port = 6006
    tb_proc = subprocess.Popen(
        [TENSORBOARD_BIN, "--logdir", str(trace_dir), "--port", str(tb_port), "--bind_all"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(3)
    tb_url = f"http://localhost:{tb_port}"
    print(f"TensorBoard running at {tb_url} (PID {tb_proc.pid})")
    print(f"Stop it later with: kill {tb_proc.pid}")
    display(Javascript(f'window.open("{tb_url}", "_blank");'))
    display(HTML(f'<a href="{tb_url}" target="_blank" rel="noopener">Open TensorBoard</a>'))
    print("In TensorBoard, open the Profile tab. If the tab is missing, confirm xprof is installed.")

### XProf trace reading checklist

| Visual clue | What it probably means | What to try next |
| ----------- | ---------------------- | ---------------- |
| Long compile span before first step | Normal first-call JIT compilation | Warm up before measuring. |
| Compile spans between steps | Recompilation | Check changing shapes, dtypes, or static args from Lesson 2. |
| GPU rows have white gaps | Host is not feeding the GPU | Look for data loading, printing, `float(loss)`, NumPy conversion. |
| Many tiny kernels | Launch overhead or too-small compiled regions | JIT a larger function; batch work. |
| Memcpy activity between steps | Host-device transfers | Keep metrics on device; log less often; use transfer guard. |
| High peak memory | Activations or temporary buffers may dominate | Open `memory_viewer`; try smaller batch. |

Other official JAX profiling entry points you should recognize: `start_trace()` / `stop_trace()`
for programmatic trace regions that do not fit neatly into a `with jax.profiler.trace(...)` block,
`start_server()` plus `python -m jax.collect_profile` for profiling long-running jobs,
and Perfetto export for opening traces in the Perfetto UI. We will not go deep on those in this
introductory lesson, but they are covered in the official
[JAX profiling guide](https://docs.jax.dev/en/latest/profiling.html).

### Remote access reminders

If you are running this notebook on a remote machine, use the table below to reach each viewer from your local device. The Nsight workflow avoids ports entirely by downloading the `.nsys-rep` artifact you'll generate later.

| Viewer | Port / artifact | How to access from your laptop |
| ------ | --------------- | ------------------------------ |
| XProf standalone | `6007` | SSH forward, e.g. `gcloud compute ssh <machine-name> -- -L 6007:localhost:6007`, then open <http://localhost:6007> |
| TensorBoard | `6006` | SSH forward, e.g. `gcloud compute ssh <machine-name> -- -L 6006:localhost:6006`, then open <http://localhost:6006> |
| Nsight Systems | `.nsys-rep` artifact | Click the **Download** link in the Nsight cell, then open the file in `nsys-ui` installed locally |

On GKE, or other managed notebook services, your environment may expose ports through its own proxy rather than direct SSH forwarding. The Nsight download flow sidesteps that entirely.

## Pitfall: host-device transfers

Pulling a JAX value back to Python inside the loop forces synchronization. Common examples are `float(loss)`, `.item()`, `np.asarray(...)`, and printing arrays.

This cell compares two logging styles. The bad version turns the loss into a Python `float` every step. The better version keeps losses as JAX arrays and synchronizes once at the end.

In [ ]:
N = 30

# Bad: convert the loss to a Python float every step.
t0 = time.perf_counter()
bad_params = params
bad_losses = []
for _ in range(N):
    bad_params, bad_loss = train_step(bad_params, batch)
    bad_losses.append(float(bad_loss))
bad_ms = (time.perf_counter() - t0) * 1000 / N

# Better: keep metrics as JAX arrays and synchronize once.
t0 = time.perf_counter()
good_params = params
good_losses = []
for _ in range(N):
    good_params, good_loss = train_step(good_params, batch)
    good_losses.append(good_loss)
jax.block_until_ready((good_params, good_losses))
good_ms = (time.perf_counter() - t0) * 1000 / N

print(f"float(loss) every step: {bad_ms:.3f} ms / step")
print(f"deferred sync:          {good_ms:.3f} ms / step")
show_bars(
    [("float(loss) every step", bad_ms), ("deferred sync", good_ms)],
    title="Cost of Pulling Metrics to Python",
    unit="ms/step",
    lower_is_better=True,
)

# Debugging helper: transfer guard can catch accidental transfers.
try:
    _, guard_loss = train_step(params, batch)
    guard_loss.block_until_ready()
    with jax.transfer_guard("disallow"):
        _ = float(guard_loss)
except RuntimeError as e:
    print("\nTransfer guard caught an implicit transfer:")
    print(str(e).splitlines()[0])

## Pitfall: inefficient batch sizes

Small batches often do not give the GPU enough parallel work. Throughput usually improves as the batch grows, then flattens once the GPU is saturated or memory becomes the limit.

Batch size also affects memory. The table below includes a simple estimate for the main batch-shaped buffers in this toy forward pass: input, hidden activation, and output. Real training uses more memory because gradients and temporary buffers also matter.

In [ ]:
@jax.jit
def forward_only(params, x):
    """Forward pass only (no loss); used here to sweep batch size and measure throughput."""
    hidden = jax.nn.gelu(x @ params["w1"])
    return hidden @ params["w2"]


batch_results = []
bytes_per_float32 = np.dtype(np.float32).itemsize

for batch_size in (1, 8, 32, 128, 256, 512, 1024):
    xb = jax.random.normal(jax.random.key(batch_size), (batch_size, IN_DIM), dtype=jnp.float32)
    _ = forward_only(params, xb).block_until_ready()  # compile/warm up this shape

    reps = 80 if batch_size <= 128 else 30
    # Async dispatch lets JAX queue all `reps` calls without blocking. We block once at the end and divide by reps,
    # so this measures *amortized* time per call when the executor stays busy. It is NOT a single-call latency:
    # add `.block_until_ready()` inside the loop for that. The examples/sec column is still meaningful as throughput.
    t0 = time.perf_counter()
    for _ in range(reps):
        yb = forward_only(params, xb)
    yb.block_until_ready()

    ms = (time.perf_counter() - t0) * 1000 / reps
    examples_per_sec = batch_size / (ms / 1000)

    # Simple memory estimate for batch-shaped forward buffers: x, hidden, and output.
    estimated_forward_mib = batch_size * (IN_DIM + HIDDEN + OUT_DIM) * bytes_per_float32 / 2**20
    batch_results.append((batch_size, ms, examples_per_sec, estimated_forward_mib))

show_table(
    ["Batch", "ms/call (avg)", "examples/sec", "estimated batch buffers"],
    [
        (batch_size, f"{ms:.3f}", f"{examples_per_sec:,.0f}", f"{estimated_mib:.1f} MiB")
        for batch_size, ms, examples_per_sec, estimated_mib in batch_results
    ],
    title="Batch Size, Throughput, and Memory",
    aligns=["right", "right", "right", "right"],
)

show_bars(
    [(f"batch {batch_size}", examples_per_sec) for batch_size, _, examples_per_sec, _ in batch_results],
    title="Throughput by Batch Size",
    unit="ex/s",
    lower_is_better=False,
)

## Pitfall: memory pressure

JAX usually preallocates 75% of GPU memory on first use. That is normal: it reduces allocation overhead and fragmentation. It also means `nvidia-smi` can look "full" even when your model is small.

Use `memory_stats()` for a quick process-level view, then XProf's memory tools for deeper analysis.

In [ ]:
def gib(value):
    """Convert a byte count to gibibytes (1 GiB = 2**30 bytes)."""
    return value / 2**30


print(f"{'device':<26} {'limit':>12} {'in use':>12} {'peak':>12}")
print("-" * 66)
for device in gpu_devices:
    stats = device.memory_stats()
    if not stats:
        print(f"{device!s:<26} memory_stats unavailable")
        continue
    limit = stats.get("bytes_limit")
    in_use = stats.get("bytes_in_use")
    peak = stats.get("peak_bytes_in_use")
    limit_s = f"{gib(limit):.2f} GiB" if limit is not None else "n/a"
    in_use_s = f"{gib(in_use):.2f} GiB" if in_use is not None else "n/a"
    peak_s = f"{gib(peak):.2f} GiB" if peak is not None else "n/a"
    print(f"{device!s:<26} {limit_s:>12} {in_use_s:>12} {peak_s:>12}")

Memory settings must be set **before importing JAX**. In a notebook, that means setting them before kernel startup and then restarting the kernel.

| Variable | Example | Use when |
| -------- | ------- | -------- |
| `XLA_PYTHON_CLIENT_MEM_FRACTION` | `0.50` | You share a GPU and want JAX to reserve less memory. |
| `XLA_PYTHON_CLIENT_PREALLOCATE` | `false` | You want on-demand allocation, accepting more fragmentation risk. |
| `XLA_PYTHON_CLIENT_ALLOCATOR` | `platform` | You are debugging memory and want deallocation; too slow for normal training. |

In [ ]:
# This cell cannot change JAX's allocator after import. It shows the current settings
# and prints the command you would use before starting the next notebook kernel.
for name in (
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    "XLA_PYTHON_CLIENT_PREALLOCATE",
    "XLA_PYTHON_CLIENT_ALLOCATOR",
):
    print(f"{name}={os.environ.get(name, '<unset>')}")

print("\nExample for a shared GPU, set before launching Python/Jupyter:")
print("export XLA_PYTHON_CLIENT_MEM_FRACTION=0.50")

## Profiling with Nsight Systems

XProf is the best first profiler for JAX. Nsight Systems is the required second view when you need the CUDA timeline: streams, CUDA API calls, kernels, memcopies, cuBLAS/cuDNN, and eventually NCCL.

### Nsight capture script

Profiling a live notebook kernel directly is not ideal, so this cell writes a small standalone Python script. The script warms up once, then uses NVTX ranges to mark the region we care about. Nsight Systems will use those ranges as landmarks in the timeline.

You do not need to read every line of the generated script. Most of it repeats the same tiny model from earlier so that `nsys` can profile a fresh Python process. **Keep the script’s `train_step` in sync with the in-notebook one** — they intentionally mirror each other, and a divergent edit to one without the other would make the Nsight timeline disagree with the XProf trace. The new idea is the NVTX annotation: it gives Nsight named regions to show in the timeline.

The workflow is:

1. Put the workload in a short script.
2. Add NVTX ranges around the steps you care about.
3. Run the script with `nsys profile`.
4. Open the `.nsys-rep` file with the Nsight Systems GUI.

In [ ]:
nsight_dir = pathlib.Path(tempfile.mkdtemp(prefix="jax-nsight-"))
script_path = nsight_dir / "nsight_train_step.py"
report_base = nsight_dir / "jax_train_step"
report_path = pathlib.Path(f"{report_base}.nsys-rep")

script_source = f"""
import nvtx
import jax
import jax.numpy as jnp

BATCH = {BATCH}
IN_DIM = {IN_DIM}
HIDDEN = {HIDDEN}
OUT_DIM = {OUT_DIM}
LR = {LR}


def init_params(key):
    k1, k2 = jax.random.split(key)
    return {{
        "w1": jax.random.normal(k1, (IN_DIM, HIDDEN), dtype=jnp.float32) * 0.02,
        "w2": jax.random.normal(k2, (HIDDEN, OUT_DIM), dtype=jnp.float32) * 0.02,
    }}


def make_batch(key):
    kx, ky = jax.random.split(key)
    return (
        jax.random.normal(kx, (BATCH, IN_DIM), dtype=jnp.float32),
        jax.random.normal(ky, (BATCH, OUT_DIM), dtype=jnp.float32),
    )


def loss_fn(params, batch):
    x, target = batch
    hidden = jax.nn.gelu(x @ params["w1"])
    pred = hidden @ params["w2"]
    return jnp.mean((pred - target) ** 2)


@jax.jit
def train_step(params, batch):
    loss, grads = jax.value_and_grad(loss_fn)(params, batch)
    params = jax.tree.map(lambda p, g: p - LR * g, params, grads)
    return params, loss


key = jax.random.key(0)
params = init_params(key)
batch = make_batch(key)

# Warm up before the region we care about.
params, loss = train_step(params, batch)
jax.block_until_ready((params, loss))

with nvtx.annotate("profile_region", domain="jax_course"):
    for step in range(8):
        with nvtx.annotate(f"train_step_{{step}}", domain="jax_course"):
            key = jax.random.fold_in(key, step)
            batch = make_batch(key)
            params, loss = train_step(params, batch)
            jax.block_until_ready((params, loss))
"""
script_path.write_text(textwrap.dedent(script_source))

print(f"Wrote script: {script_path}")
print(f"Report path:  {report_path}")

### Run Nsight Systems

This command starts collection only when the NVTX range `profile_region@jax_course` begins, then stops when that range ends. That keeps the report focused on post-warmup execution.

In [ ]:
import base64

# This capture starts at the NVTX range profile_region@jax_course and stops when it ends.
nsys_cmd = [
    NSYS_BIN,
    "profile",
    "--trace=cuda,nvtx,osrt,cudnn,cublas",
    "--capture-range=nvtx",
    "--capture-range-end=stop",
    "--nvtx-capture=profile_region@jax_course",
    "--force-overwrite=true",
    f"--output={report_base}",
    "-e",
    "NSYS_NVTX_PROFILER_REGISTER_ONLY=0",
    sys.executable,
    str(script_path),
]

print("Running Nsight Systems:")
print(" ".join(nsys_cmd))
nsys_result = subprocess.run(nsys_cmd, capture_output=True, text=True, check=False)

if nsys_result.returncode != 0:
    print("nsys profile failed")
    print(nsys_result.stdout[-2000:])
    print(nsys_result.stderr[-4000:])
    raise RuntimeError("Nsight Systems capture failed. This lesson requires nsys to run successfully.")

print(f"Nsight report: {report_path}  ({report_path.stat().st_size / 1024:.1f} KB)")

b64 = base64.b64encode(report_path.read_bytes()).decode()
download_html = (
    f'<a href="data:application/octet-stream;base64,{b64}" '
    f'download="{report_path.name}" '
    f'style="font-weight:600;">Download {report_path.name}</a>'
)

display(HTML(download_html))

### Reading the Nsight timeline

> **To open this report in Nsight Systems GUI**
>
> 1. Install Nsight Systems on your local machine from [developer.nvidia.com/nsight-systems](https://developer.nvidia.com/nsight-systems) (free, Linux/macOS/Windows).
> 2. Download the generated `.nsys-rep` report from the notebook.
> 3. Launch `nsys-ui` and open the downloaded `.nsys-rep` file with **File -> Open**, or drag and drop it into the Nsight Systems window.

Start with these rows:

| Nsight row | What to inspect |
| ---------- | --------------- |
| NVTX | Find `profile_region` and `train_step_*`. These are your map. |
| CUDA GPU rows | Look for kernel coverage and white gaps. White gaps mean idle GPU time. |
| CUDA API | Look for `cudaLaunchKernel`, `cudaMemcpy*`, and synchronization calls. |
| cuBLAS / cuDNN | Matmul, convolution, and attention library calls appear here when traced. |
| OS Runtime (`osrt`) | Host waits, locks, sleeps, and other blocking behavior. |

For GPU utilization metrics, rerun with `--gpu-metrics-devices=cuda-visible` if your environment allows GPU metrics collection.

### Summarize the Nsight report in the notebook

The GUI is the main Nsight experience, but `nsys stats` gives a useful text summary. We will print the top kernel and memory-operation tables.

In [ ]:
stats_cmd = [
    NSYS_BIN,
    "stats",
    "--quiet",
    "--report",
    "cuda_gpu_kern_sum,cuda_gpu_mem_time_sum,cuda_api_sum",
    "--format",
    "csv",
    "--timeunit",
    "msec",
    str(report_path),
]

stats_result = subprocess.run(stats_cmd, capture_output=True, text=True, check=False)
if stats_result.returncode != 0:
    print("nsys stats failed")
    print(stats_result.stderr[-3000:])
else:
    sections = [s for s in stats_result.stdout.strip().split("\n\n") if s.strip()]
    for idx, section in enumerate(sections[:3], start=1):
        reader = csv.DictReader(io.StringIO(section))
        rows = list(reader)
        if not rows:
            continue
        headers = reader.fieldnames or []
        print(f"\nReport section {idx}: {headers}")
        compact_rows = []
        for row in rows[:8]:
            name = row.get("Name") or row.get("Operation") or row.get("Name:Demangled") or ""
            time_pct = row.get("Time (%)", "")
            total = row.get("Total Time (ms)") or row.get("Total Time (msec)") or row.get("Total Time") or row.get("Total Time (ns)") or ""
            count = row.get("Instances") or row.get("Count") or row.get("Num Calls") or row.get("Calls") or ""
            compact_rows.append((time_pct, total, count, name[:90]))
        show_table(["Time (%)", "Total time (ms)", "Count", "Name / operation"], compact_rows, title=f"Nsight stats section {idx}", aligns=["right", "right", "right", "left"])

## Summary

The goal of this lesson was to give you a practical first debugging path for JAX on GPU. When performance looks strange, resist the urge to guess. Start with honest timing, capture a trace, and use the visual clues to decide what to investigate next.

You now practiced the profiling workflow:

* Use `block_until_ready()` for honest timing.
* Separate compile time from cached execution time.
* Use `jax.profiler.trace` and XProf/TensorBoard to understand JAX-level execution.
* Look first for recompilation, too many small operations, host-device transfers, inefficient batches, and memory pressure.
* Use `XLA_PYTHON_CLIENT_MEM_FRACTION` before startup when you need to reduce JAX's GPU memory reservation.
* Use Nsight Systems when you need the CUDA timeline.

The rule of thumb: measure first, then optimize the specific pattern the trace shows.